In [1]:
# --- BLOQUE 0: INFRAESTRUCTURA ---
import sys
import subprocess
import warnings

warnings.filterwarnings('ignore')

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package])

try:
    from astroquery.heasarc import Heasarc
    import astropy
    import scipy
    import matplotlib
except ImportError:
    print("[SYSTEM] Desplegando Astroquery...")
    install("astroquery")
    install("astropy")
    install("scipy")
    install("matplotlib")
    from astroquery.heasarc import Heasarc

import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo
from scipy.optimize import curve_fit

print("[SYSTEM] Iniciando Protocolo ASTRO-MINER (Fixed Query)...")

# --- BLOQUE 1: EXTRACCIÓN (MÉTODO COMPATIBLE) ---
heasarc = Heasarc()
TARGET_TABLE = 'fermilgrb'

print(f"[QUERY] Descargando tabla completa '{TARGET_TABLE}'...")

try:
    # CORRECCIÓN TÉCNICA:
    # Usamos query_table para traer todos los objetos de la tabla 'fermilgrb'.
    # fields='All' asegura que vengan TODAS las columnas (Redshift, Energía, etc).
    # Esto evita el error de "unexpected keyword argument" y el de "column missing".
    table = heasarc.query_table(TARGET_TABLE, fields='All', result_max=3000)

    df = table.to_pandas()
    # Normalizar columnas a mayúsculas
    df.columns = [c.upper() for c in df.columns]
    print(f"[SUCCESS] Tabla recuperada. Registros: {len(df)}")

except Exception as e:
    print(f"[CRITICAL] Error de conexión HEASARC: {e}")
    sys.exit()

# --- BLOQUE 2: MAPEO Y FILTRADO ---
# Buscamos las columnas que TÚ confirmaste que existen en el volcado de texto.
# REDSHIFT: Distancia
# TL0: Tiempo de inicio de emisión LAT (el mejor proxy para llegada de alta energía)
# NAME: Nombre

try:
    raw_z = df['REDSHIFT']
    # Usamos TL0 como el tiempo de llegada de los fotones de alta energía
    # (El texto dice: "Tl0: El momento en que el primer fotón... se detecta")
    raw_t = df['TL0']
    raw_n = df['NAME']

    # Para la energía, intentamos buscar la columna de energía máxima si existe,
    # si no, asumimos un valor medio de LAT (>100 MeV) para el plot cualitativo.
    if 'EXT_EMISSION_MAX_ENE' in df.columns:
        raw_e = df['EXT_EMISSION_MAX_ENE'] / 1000.0 # MeV -> GeV
    else:
        # Fallback: Usar Like_Best_Flux_Ene o simplemente asumir escala GeV para ver la tendencia temporal
        # Pero 'TL0' es específico de LAT (>100 MeV), así que ya es "alta energía".
        print("[INFO] Usando energía promedio de banda LAT para normalización.")
        raw_e = np.ones(len(df)) * 1.0 # 1 GeV nominal para ver la tendencia temporal pura

except KeyError as e:
    print(f"[ERROR] Columna no encontrada en el dataframe: {e}")
    print(f"Columnas disponibles: {list(df.columns)}")
    sys.exit()

# Filtros Físicos
# 1. Redshift definido (> 0)
# 2. Detección LAT válida (TL0 existe)
mask = (raw_z > 0) & (np.isfinite(raw_z)) & (np.isfinite(raw_t))

clean_z = raw_z[mask].values
clean_t = raw_t[mask].values
clean_e = raw_e[mask].values if isinstance(raw_e, np.ndarray) else raw_e[mask].values
clean_names = raw_n[mask].values

print(f"[DATA] Eventos con Redshift y Tiempos LAT: {len(clean_z)}")

if len(clean_z) < 3:
    print("[WARN] Pocos datos. ¿Tabla vacía?")

# --- BLOQUE 3: FÍSICA (LIV TEST) ---
print("[PHYS] Calculando distancias cosmológicas...")

# K-Factor (Distancia relativista)
def get_k(z):
    z_grid = np.linspace(0, z, 50)
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    integrand = (1+z_grid) / Ez
    return np.trapz(integrand, z_grid)

if len(clean_z) > 0:
    K_factors = np.array([get_k(z) for z in clean_z])
    # Observable: Retraso temporal (s)
    # Si hay LIV, t_delay = (E/M_QG)*K.
    # Aquí graficamos t vs K. Si vemos pendiente, hay efecto.
    lags = clean_t
else:
    K_factors = []
    lags = []

# --- BLOQUE 4: VISUALIZACIÓN ---
plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

if len(K_factors) > 0:
    # Scatter
    plt.scatter(K_factors, lags, c=lags, cmap='inferno', s=80, alpha=0.9, label='Datos Fermi-LAT')
    plt.colorbar(label='Retraso TL0 (s)')

    # Ajuste Lineal Simple
    def model(k, m, c): return m * k + c

    if len(K_factors) > 2:
        try:
            # Robust fit (quitando outliers extremos > 100s)
            mask_fit = lags < 100
            popt, pcov = curve_fit(model, K_factors[mask_fit], lags[mask_fit])
            m_fit = popt[0]

            x_plot = np.linspace(0, max(K_factors)*1.1, 50)
            plt.plot(x_plot, model(x_plot, *popt), color='#00ffcc',
                     linestyle='--', lw=2, label=f'Tendencia (m={m_fit:.2f})')
        except: m_fit = 0

    plt.axhline(0, color='white', linestyle=':', label='Simultaneidad (Einstein)')

    # Etiquetas
    for i in range(len(clean_names)):
        # Etiquetar outliers interesantes
        if K_factors[i] > 2 or lags[i] > 20:
            try: n_str = str(clean_names[i])
            except: n_str = "GRB"
            plt.annotate(n_str, (K_factors[i], lags[i]), color='yellow', fontsize=8)

plt.xlabel(r'Factor de Distancia Cosmológica $K(z)$')
plt.ylabel(r'Tiempo de Inicio LAT $T_{L0}$ (s)')
plt.title('Test de Gravedad Cuántica: ¿Llegan tarde los fotones?\nDatos: HEASARC fermilgrb (Columna TL0)', color='white')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if 'm_fit' in locals() and m_fit > 0.5:
    print(f"\n[RESULT] Pendiente positiva detectada (m={m_fit:.2f}).")
    print(">> POSIBLE SEÑAL: Los GRBs lejanos tardan más en encenderse en alta energía.")
else:
    print("\n[RESULT] Sin correlación evidente con la distancia.")

[SYSTEM] Desplegando Astroquery...
[SYSTEM] Iniciando Protocolo ASTRO-MINER (Fixed Query)...
[QUERY] Descargando tabla completa 'fermilgrb'...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



[CRITICAL] Error de conexión HEASARC: Unable to find coordinates for name 'GRB%' using https://cds.unistra.fr/cgi-bin/nph-sesame/SNV?GRB%25
Traceback (most recent call last):
  File "/tmp/ipython-input-520908613.py", line 42, in <cell line: 0>
    table = heasarc.query_object(object_name='GRB%', mission=TARGET_TABLE,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astropy/utils/decorators.py", line 143, in deprecated_func
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astroquery/heasarc/core.py", line 463, in query_object
    pos = coordinates.SkyCoord.from_name(object_name)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astropy/coordinates/sky_coordinate.py", line 1905, in from_name
    icrs_coord = get_icrs_coordinates(name, parse, cache=cache)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

TypeError: object of type 'NoneType' has no len()